**This notebook is an exercise in the [SQL](https://www.kaggle.com/learn/intro-to-sql) course.  You can reference the tutorial at [this link](https://www.kaggle.com/dansbecker/group-by-having-count).**

---


# Introduction

Queries with **GROUP BY** can be powerful. There are many small things that can trip you up (like the order of the clauses), but it will start to feel natural once you've done it a few times. Here, you'll write queries using **GROUP BY** to answer questions from the Hacker News dataset.

Before you get started, run the following cell to set everything up:

In [1]:
# Set up feedback system
from learntools.core import binder
binder.bind(globals())
from learntools.sql.ex3 import *
print("Setup Complete")

Using Kaggle's public dataset BigQuery integration.


/usr/local/lib/python3.11/dist-packages/google/cloud/bigquery/table.py:1727: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Setup Complete


The code cell below fetches the `full` table from the `hacker_news` dataset.  We also preview the first five rows of the table.

In [2]:
from google.cloud import bigquery

# Create a "Client" object
client = bigquery.Client()

# Construct a reference to the "hacker_news" dataset
dataset_ref = client.dataset("hacker_news", project="bigquery-public-data")

# API request - fetch the dataset
dataset = client.get_dataset(dataset_ref)

# Construct a reference to the "full" table
table_ref = dataset_ref.table("full")

# API request - fetch the table
table = client.get_table(table_ref)

# Preview the first five lines of the table
client.list_rows(table, max_results=10).to_dataframe()

Using Kaggle's public dataset BigQuery integration.


,title,url,text,dead,by,score,time,timestamp,type,id,parent,descendants,ranking,deleted
0,None,None,None,<NA>,None,<NA>,<NA>,NaT,story,99390,<NA>,<NA>,<NA>,<NA>
1,None,None,None,<NA>,None,<NA>,1437692677,2015-07-23 23:04:37+00:00,story,9939041,<NA>,<NA>,<NA>,<NA>
2,None,None,None,<NA>,None,<NA>,1437692820,2015-07-23 23:07:00+00:00,story,9939054,<NA>,<NA>,<NA>,<NA>
3,None,None,None,<NA>,None,<NA>,<NA>,NaT,story,9939062,<NA>,<NA>,<NA>,<NA>
4,None,None,None,<NA>,None,<NA>,1437695628,2015-07-23 23:53:48+00:00,story,9939234,<NA>,<NA>,<NA>,<NA>
5,None,None,None,<NA>,None,<NA>,<NA>,NaT,story,9939272,<NA>,<NA>,<NA>,<NA>
6,None,None,None,<NA>,None,<NA>,1437696513,2015-07-24 00:08:33+00:00,story,9939292,<NA>,<NA>,<NA>,<NA>
7,None,None,None,<NA>,None,<NA>,1437698381,2015-07-24 00:39:41+00:00,story,9939406,<NA>,<NA>,<NA>,<NA>
8,None,None,None,<NA>,None,<NA>,<NA>,NaT,story,9939414,<NA>,<NA>,<NA>,<NA>
9,None,None,None,<NA>,None,<NA>,1260788681,2009-12-14 11:04:41+00:00,story,993949,<NA>,<NA>,<NA>,<NA>


In [3]:
list_tables = list(client.list_tables(dataset))
for table_name in list_tables:
    print(table.table_id)

table_ref = dataset_ref.table("full")
table = client.get_table(table_ref)
df = client.list_rows(table, max_results = 5).to_dataframe()

for row in table.schema:
    print(row.name, row.field_type)

print("Null values: ",df.isnull().sum().sum())


full
title STRING
url STRING
text STRING
dead BOOLEAN
by STRING
score INTEGER
time INTEGER
timestamp TIMESTAMP
type STRING
id INTEGER
parent INTEGER
descendants INTEGER
ranking INTEGER
deleted BOOLEAN
Null values:  54


# Exercises

### 1) Prolific commenters

Hacker News would like to send awards to everyone who has written more than 10,000 posts. Write a query that returns all authors with more than 10,000 posts as well as their post counts. Call the column with post counts `NumPosts`.

In case sample query is helpful, here is a query you saw in the tutorial to answer a similar question:
```
query = """
        SELECT parent, COUNT(1) AS NumPosts
        FROM `bigquery-public-data.hacker_news.full`
        GROUP BY parent
        HAVING COUNT(1) > 10
        """
```

In [4]:
# Query to select prolific commenters and post counts
prolific_commenters_query = """
                            SELECT `by` as author, COUNT(1) as NumPosts
                            FROM `bigquery-public-data.hacker_news.full`
                            GROUP BY author
                            Having COUNT(1) > 10000
""" # Your code goes here

# Set up the query (cancel the query if it would use too much of 
# your quota, with the limit set to 1 GB)
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=10**10)
query_job = client.query(prolific_commenters_query, job_config=safe_config)

# API request - run the query, and return a pandas DataFrame
prolific_commenters = query_job.to_dataframe()

# View top few rows of results
print(prolific_commenters.head())

# Check your answer
q_1.check()

/usr/local/lib/python3.11/dist-packages/google/cloud/bigquery/table.py:1727: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


       author  NumPosts
0   agumonkey     21571
1  prostoalex     11526
2  Brajeshwar     15660
3     bombcar     20773
4        jerf     17552


<IPython.core.display.Javascript object>

<span style="color:#33cc33">Correct</span>

For the solution, uncomment the line below.

In [5]:
q_1.solution()

<IPython.core.display.Javascript object>

<span style="color:#33cc99">Solution:</span> 
```python

prolific_commenters_query = """
                            SELECT `by` AS author, COUNT(1) AS NumPosts
                            FROM `bigquery-public-data.hacker_news.full`
                            GROUP BY author
                            HAVING COUNT(1) > 10000
                            """

```

### 2) Deleted comments

How many comments have been deleted? (If a comment was deleted, the `deleted` column in the table will have the value `True`.)

In [6]:
# Write your query here and figure out the answer

In [7]:
deleted_posts_query = """SELECT COUNT(1) as num_deleted_posts
                        FROM `bigquery-public-data.hacker_news.full`
                        WHERE deleted = True
""" # Put your answer here
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed = 10**10)
query_job = client.query(deleted_posts_query, job_config = safe_config)
deleted_posts = query_job.to_dataframe()
num_deleted_posts = deleted_posts['num_deleted_posts'][0]
# Check your answer
q_2.check()

/usr/local/lib/python3.11/dist-packages/google/cloud/bigquery/table.py:1727: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


<IPython.core.display.Javascript object>

<span style="color:#33cc33">Correct</span>

For the solution, uncomment the line below.

In [8]:
# q_2.solution()

# Keep Going
**[Click here](https://www.kaggle.com/dansbecker/order-by)** to move on and learn about the **ORDER BY** clause.

---




*Have questions or comments? Visit the [course discussion forum](https://www.kaggle.com/learn/intro-to-sql/discussion) to chat with other learners.*